# Lab 4 — Build the Audience, Then Buy It
Name: Dzhura Daryna | Group: КМ-41

## Setup

In [1]:
!pip install -q kagglehub pyarrow

import re
import numpy as np
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


## Task 1.0 — Clean the survey file

In [2]:
audit = []
s = survey.copy()
print("Survey data rows:", len(s) - 3)

m = (s.RESPID.str.strip().str.upper() == "RESPID") | (s.KEYWORD.str.strip().str.upper() == "KEYWORD")
audit.append(("repeated header rows", int(m.sum()), "dropped")); s = s[~m]
m = s.RESPID.str.strip().str.upper().str.startswith("TOTAL")
audit.append(("trailer row 'TOTAL ROWS'", int(m.sum()), "dropped")); s = s[~m]

raw_id = s.RESPID
s["RESPID"] = raw_id.str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
audit.append(("ids with spaces, '.0' or leading 0", int((s.RESPID != raw_id).sum()), "cleaned"))
bad = ~s.RESPID.str.fullmatch(r"\d{8}")
audit.append(("malformed ids", int(bad.sum()), "dropped")); s = s[~bad]

raw_kw = s.KEYWORD
s["KEYWORD"] = raw_kw.str.strip().str.upper()
audit.append(("keywords with wrong case/spaces", int((s.KEYWORD != raw_kw).sum()), "normalized"))

v = s.VALUE.str.strip().str.lower()
yes = v.isin(["1", "1.0", "y", "yes", "true"])
audit.append(("yes written as 1.0/y/yes/TRUE", int((yes & (v != "1")).sum()), "treated as yes"))
audit.append(("non-yes values (0/N/no/blank)", int((~yes).sum()), "dropped")); s = s[yes]

layout = survey_layout[survey_layout.RECTYPE == "B"].copy()
layout["KEYWORD"] = layout.KEYWORD.str.strip().str.upper()
layout["SAMPLE"] = layout.SAMPLE.astype(int)
ok = s.KEYWORD.isin(layout.KEYWORD)
audit.append(("codes not in layout", int((~ok).sum()), "dropped")); s = s[ok]

d = s.duplicated(["RESPID", "KEYWORD"])
audit.append(("duplicate person+keyword", int(d.sum()), "kept one")); s = s[~d]

va = viewer_attributes.copy()
va["respondent_id"] = va.respondent_id.astype(str)
va["attr_id"] = pd.to_numeric(va.attr_id)
rl = set(va[va.attr_id.isin([101, 102])].respondent_id)
age_s = pd.to_numeric(va[va.attr_id == 300].set_index("respondent_id").value, errors="coerce")
adults = sorted(set(age_s[age_s >= 18].index) & rl)
na = ~s.RESPID.isin(adults)
audit.append(("ids not adult panel members", int(na.sum()), "dropped")); s = s[~na]

print(pd.DataFrame(audit, columns=["kind", "rows", "action"]).to_string(index=False))
yes_clean = s
print("Clean yes answers:", len(yes_clean))
print("Adult panel members:", len(adults))

Survey data rows: 18346
                              kind  rows         action
              repeated header rows     2        dropped
          trailer row 'TOTAL ROWS'     1        dropped
ids with spaces, '.0' or leading 0  1455        cleaned
                     malformed ids     0        dropped
   keywords with wrong case/spaces  1104     normalized
     yes written as 1.0/y/yes/TRUE  1194 treated as yes
     non-yes values (0/N/no/blank)   742        dropped
               codes not in layout   738        dropped
          duplicate person+keyword   952       kept one
       ids not adult panel members   404        dropped
Clean yes answers: 15510
Adult panel members: 8010


In [3]:
people = pd.DataFrame(index=pd.Index(adults, name="RESPID"))
kw = yes_clean.groupby("RESPID").KEYWORD.apply(set)
people["kw"] = kw.reindex(people.index).apply(lambda x: x if isinstance(x, set) else set())
pair_map = {"X710101": 1, "X710102": 2, "X710103": 3}
people["n_pair_codes"] = people.kw.apply(lambda k: sum(c in pair_map for c in k))
people["pairs_num"] = people.kw.apply(lambda k: max([pair_map[c] for c in k if c in pair_map], default=None))
for b, c in {"stride": "X710201", "kestrel": "X710202", "easywalk": "X710203"}.items():
    people[b] = people.kw.apply(lambda k, c=c: c in k)
people["buyer"] = people.kw.apply(lambda k: len(k) > 0)
people["pairs"] = people.pairs_num.map({1: "1", 2: "2", 3: "3+"})
people.loc[~people.buyer, "pairs"] = "0"

print("People with two pair counts:", int((people.n_pair_codes > 1).sum()))
print("Non-buyers:", int((~people.buyer).sum()))

mine = {"X710100": int(people.buyer.sum()), "X710101": int((people.pairs == "1").sum()),
        "X710102": int((people.pairs == "2").sum()), "X710103": int((people.pairs == "3+").sum()),
        "X710201": int(people.stride.sum()), "X710202": int(people.kestrel.sum()),
        "X710203": int(people.easywalk.sum()),
        "X710298": yes_clean[yes_clean.KEYWORD == "X710298"].RESPID.nunique(),
        "X710299": yes_clean[yes_clean.KEYWORD == "X710299"].RESPID.nunique()}
check = layout[["KEYWORD", "Text", "SAMPLE"]].copy()
check["my_count"] = check.KEYWORD.map(mine); check["diff"] = check.my_count - check.SAMPLE
print(check.to_string(index=False))

raw = survey.iloc[:-1]; raw = raw[raw.RESPID != "RESPID"]
raw_kestrel = int(((raw.KEYWORD == "X710202") & (raw.VALUE == "1")).sum())
raw_buyers = raw[(raw.KEYWORD == "X710100") & (raw.VALUE == "1")].RESPID.nunique()
print("Kestrel: raw", raw_kestrel, "| clean", int(people.kestrel.sum()),
      "| diff %", round((raw_kestrel - people.kestrel.sum()) / people.kestrel.sum() * 100, 1))
print("Buyers share: raw", round(raw_buyers / raw.RESPID.nunique(), 3),
      "| clean", round(people.buyer.mean(), 3))

People with two pair counts: 113
Non-buyers: 3231
KEYWORD            Text  SAMPLE  my_count  diff
X710100             Any    4779      4779     0
X710101          1 pair    1810      1810     0
X710102         2 pairs    1308      1308     0
X710103 3 or more pairs    1661      1661     0
X710201          STRIDE    1563      1563     0
X710202         KESTREL     354       354     0
X710203        EASYWALK    1050      1050     0
X710298     OTHER BRAND    2739      2739     0
X710299      DON'T KNOW     371       371     0
Kestrel: raw 383 | clean 354 | diff % 8.2
Buyers share: raw 0.569 | clean 0.597


**Answer:** The file had eight kinds of bad records: 2 repeated header rows and 1 trailer line "TOTAL ROWS" (dropped); 1,455 ids with spaces, a ".0" suffix or a leading zero (cleaned to the 8-digit id); 1,104 keywords in lower case or with spaces (upper-cased, trimmed); 1,194 yes answers written as 1.0 / y / Y / yes / TRUE (mapped to yes); 742 non-yes values (0, 0.0, N, no, blank), dropped because the export lists only yes answers; 738 codes that are not in the layout (dropped); 952 duplicate person+keyword rows (kept one); 404 rows of people who are not adult panel members (dropped). 113 people had two pair counts and kept the higher one. After cleaning, all nine KEYWORD counts equal the layout SAMPLE (difference 0). Used as delivered, the Kestrel count would be 383 instead of 354 (+8.2%), and the buyer share 0.569 instead of 0.597.

## Task 1.1 — Four definitions

In [4]:
male = set(va[va.attr_id == 201].respondent_id); female = set(va[va.attr_id == 202].respondent_id)
people["age"] = age_s.reindex(people.index)
people["sex"] = ["M" if i in male else "F" if i in female else None for i in people.index]
people["heavy"] = people.pairs == "3+"
people["age_grp"] = pd.cut(people.age, [17, 34, 54, 200], labels=["18-34", "35-54", "55+"])

dw = daily_weights[daily_weights.resp_type.isin(["R", "L"])].copy()
dw["RESPID"] = dw.respondent_id.astype(str)
dw = dw.merge(people[["age", "sex", "buyer", "heavy", "kestrel", "age_grp"]], left_on="RESPID", right_index=True)
days = sorted(dw.broadcast_date.unique())

def size(mask):
    return dw[mask].groupby("broadcast_date").weight.sum().reindex(days, fill_value=0).mean() / 1e6

aud = {"Adults 18+": dw.age >= 18, "Shoe buyers": dw.buyer, "Heavy buyers 3+": dw.heavy,
       "Kestrel buyers": dw.kestrel, "Adults 25-54": dw.age.between(25, 54)}
print("Adults 18+ (M):", round(size(aud["Adults 18+"]), 1))
print("Shoe buyers (M):", round(size(aud["Shoe buyers"]), 1))
print("Heavy buyers, 3+ pairs (M):", round(size(aud["Heavy buyers 3+"]), 1))
print("Kestrel buyers (M):", round(size(aud["Kestrel buyers"]), 1))
print("Adults 25-54 (M):", round(size(aud["Adults 25-54"]), 1))
print("Kestrel buyers in panel (n):", int(people.kestrel.sum()))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [5]:
groups6 = {f"{s_} {a_}": (dw.sex == s_) & (dw.age_grp == a_) for s_ in ["M", "F"] for a_ in ["18-34", "35-54", "55+"]}
base6 = {g: size(aud["Adults 18+"] & m) / size(aud["Adults 18+"]) for g, m in groups6.items()}
rows = []
for name, a in aud.items():
    tot = size(a)
    for g, m in groups6.items():
        sh = size(a & m) / tot
        rows.append((name, g, round(sh, 3), round(sh / base6[g] * 100, 1)))
prof6 = pd.DataFrame(rows, columns=["audience", "group", "share", "index"])
print("SHARE"); print(prof6.pivot(index="audience", columns="group", values="share").to_string())
print("INDEX"); print(prof6.pivot(index="audience", columns="group", values="index").to_string())

SHARE
group            F 18-34  F 35-54  F 55+  M 18-34  M 35-54  M 55+
audience                                                         
Adults 18+         0.144    0.160  0.208    0.144    0.160  0.185
Adults 25-54       0.174    0.326  0.000    0.174    0.326  0.000
Heavy buyers 3+    0.139    0.226  0.196    0.111    0.138  0.189
Kestrel buyers     0.181    0.265  0.121    0.118    0.189  0.126
Shoe buyers        0.140    0.178  0.197    0.140    0.158  0.186
INDEX
group            F 18-34  F 35-54  F 55+  M 18-34  M 35-54  M 55+
audience                                                         
Adults 18+         100.0    100.0  100.0    100.0    100.0  100.0
Adults 25-54       121.2    204.1    0.0    121.2    204.1    0.0
Heavy buyers 3+     97.0    141.9   94.0     77.1     86.8  102.2
Kestrel buyers     126.0    165.8   58.0     82.1    118.6   68.2
Shoe buyers         97.6    111.4   94.6     97.4     99.2  100.8


**Answer:** I would give the planner heavy buyers (3+ pairs). It is the marketing team's own definition, it is large (49.6M people, 19.6% of adults) and it rests on 1,661 panelists, so program ratings built on it are stable (Tasks 2 and 4). Its profile is distinctive: women 35-54 index 141.9, men 18-34 only 77.1. Kestrel buyers (10.0M, 4.0% of adults) are only 354 panelists, so ratings on them are noisy (Task 2.3), and they only describe current buyers. Adults 25-54 is the wrong target: it drops everyone 55+, who are 38.5% of heavy buyers, and it pays for non-buyers (index 204 for women and men 35-54 is only age, not buying).

"Athletic-shoe buyers" look almost like all adults because 58.6% of adults (148.1M of 252.6M) bought at least one pair: an audience that wide is nearly the population. Its index stays between 94.6 and 111.4 in all six age-sex groups.

A fused purchase attribute cannot tell us whether a program's viewers buy more than other people of the same age, sex and income. In the fusion, buying depends on viewing only through demographics, so differences in a program's index only mirror its demographics, not real buying behavior. It also cannot tell whether an ad changes what people buy.

## Task 2.1 — Programs by audience

In [6]:
TARGETS = [("NFL Football", "NBC", "Sunday Night Football (NBC)"), ("NFL Football", "ABC", "Monday Night Football (ABC)"),
           ("Dancing with the Stars", "ABC", "Dancing with the Stars (ABC)"), ("NCIS", "CBS", "NCIS (CBS)"),
           ("Hannity", "FNC", "Hannity (FNC)"), ("Unsellable Houses", "HGTV", "Unsellable Houses (HGTV)"),
           ("Halloween Wars", "FOOD", "Halloween Wars (FOOD)"), ("Grotesquerie", "FX", "Grotesquerie (FX)")]
tmap = pd.DataFrame(TARGETS, columns=["program_name", "network_code", "label"])
tel = programs[programs.daypart == "Prime"].merge(tmap, on=["program_name", "network_code"])
tel["broadcast_date"] = tel.broadcast_date.astype(str)

dwx = dw.copy(); dwx["broadcast_date"] = dwx.broadcast_date.astype(str)
v = viewings[viewings.resp_type.isin(["R", "L"]) & viewings.network_code.isin(tel.network_code.unique())]
v = v[["respondent_id", "network_code", "start_ts", "end_ts"]]
x = (tel[["telecast_id", "label", "network_code", "broadcast_date", "start_ts", "end_ts"]]
     .rename(columns={"start_ts": "t0", "end_ts": "t1"}).merge(v, on="network_code"))
x = x[(x.start_ts < x.t1) & (x.end_ts > x.t0)]
x["mins"] = ((x[["end_ts", "t1"]].min(axis=1) - x[["start_ts", "t0"]].max(axis=1)).dt.total_seconds() / 60)
x = x.merge(dwx[["respondent_id", "broadcast_date", "weight", "age", "heavy", "kestrel"]], on=["respondent_id", "broadcast_date"])
x["wm"] = x.weight * x.mins

AUD = {"adults": aud["Adults 18+"], "heavy": aud["Heavy buyers 3+"], "kestrel": aud["Kestrel buyers"]}
XM = {"adults": x.age >= 18, "heavy": x.heavy, "kestrel": x.kestrel}
dsz = {k: dw[m].groupby(dw.broadcast_date.astype(str)).weight.sum() for k, m in AUD.items()}
res = tel[["telecast_id", "label", "broadcast_date"]].copy()
res["dur"] = (tel.end_ts - tel.start_ts).dt.total_seconds().div(60).values
for k in AUD:
    wm = x[XM[k]].groupby("telecast_id").wm.sum().reindex(res.telecast_id, fill_value=0).values
    res[f"imp_{k}"] = wm / res.dur
    res[f"rat_{k}"] = res[f"imp_{k}"] / res.broadcast_date.map(dsz[k]) * 100
g = res.groupby("label")[[c for c in res.columns if c.startswith(("imp_", "rat_"))]].mean()
out = pd.DataFrame({"adults rating": g.rat_adults.round(2), "target rating": g.rat_heavy.round(2),
                    "target viewers (k)": (g.imp_heavy / 1e3).round(1),
                    "composition": (g.imp_heavy / g.imp_adults).round(3),
                    "index": (g.rat_heavy / g.rat_adults * 100).round(1),
                    "kestrel index": (g.rat_kestrel / g.rat_adults * 100).round(1)})

def to_min(t):
    h, m = str(t).split(":"); return int(h) * 60 + int(m)
def cost_of(row):
    p = rate_card[(rate_card.network_code == row.network_code) & (rate_card.program_name == row.program_name)]
    if len(p) == 1: return p.cost_30s_usd.iloc[0], "program"
    day0 = pd.Timestamp(row.broadcast_date); off = (row.start_ts - day0).total_seconds() / 60; wd = day0.weekday()
    t = rate_card[(rate_card.network_code == row.network_code) & rate_card.program_name.isna()]
    hit = [r.cost_30s_usd for _, r in t.iterrows()
           if wd in [int(c) for c in re.findall(r"\d", str(r.weekdays))] and to_min(r.start_time) <= off < to_min(r.end_time)]
    return (hit[0] if len(hit) == 1 else None), f"time x{len(hit)}"
cc = tel.apply(cost_of, axis=1, result_type="expand"); tel["cost"], tel["how"] = cc[0], cc[1]
print(tel.groupby(["label", "how"]).cost.agg(["count", "mean"]))
assert tel.cost.notna().all()

res["cost"] = res.telecast_id.map(tel.set_index("telecast_id").cost)
g2 = res.groupby("label").agg(cost=("cost", "mean"), imp_adults=("imp_adults", "mean"), imp_heavy=("imp_heavy", "mean"))
out["cost per :30 ($)"] = g2.cost.round(0)
out["adults CPM ($)"] = (g2.cost / g2.imp_adults * 1000).round(2)
out["target CPM ($)"] = (g2.cost / g2.imp_heavy * 1000).round(2)
print(out.to_string())
print("Target universe (M):", round(size(aud["Heavy buyers 3+"]), 1))

                                      count      mean
label                        how                     
Dancing with the Stars (ABC) time x1      9  158600.0
Grotesquerie (FX)            time x1     25    7592.0
Halloween Wars (FOOD)        time x1      9    9700.0
Hannity (FNC)                time x1     22   39900.0
Monday Night Football (ABC)  program      3  723600.0
NCIS (CBS)                   time x1      3  178800.0
Sunday Night Football (NBC)  program      4  487200.0
Unsellable Houses (HGTV)     time x1     13   12900.0
                              adults rating  target rating  target viewers (k)  composition  index  kestrel index  cost per :30 ($)  adults CPM ($)  target CPM ($)
label                                                                                                                                                              
Dancing with the Stars (ABC)           3.25           3.14              1564.2        0.191   96.8           82.7          158600.0 

## Task 2.2 — Index, volume or price?

In [7]:
print("Highest index (program):", out["index"].idxmax())
print("Most target viewers (program):", out["target viewers (k)"].idxmax())
print("Lowest target CPM (program):", out["target CPM ($)"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** No, not the same program on every measure. Unsellable Houses (HGTV) has both the highest index (126.8) and the lowest target CPM ($40.66), but it delivers only 317k target viewers per spot. Monday Night Football (ABC) delivers the most (4,149k) but at a target CPM of $174.40, the highest of the eight. To spend $1M on Unsellable Houses alone we would need about 77 spots, far more than the 3 or so it airs in a week. To a client who wants "only the highest index" I would say what Lecture 5's shampoo example says: index measures concentration, not volume. A tiny program can have a high index and still deliver too few buyers, and even at index 126.8, 75% of its viewers (composition 0.249) are not heavy buyers. We should buy on target CPM, then add enough volume (and reach) to hit the goal.

## Task 2.3 — Kestrel buyers

In [8]:
t23 = out[["index", "kestrel index"]].rename(columns={"index": "heavy index"})
t23["heavy panelists watching"] = x[x.heavy].groupby("label").respondent_id.nunique()
t23["kestrel panelists watching"] = x[x.kestrel].groupby("label").respondent_id.nunique()
print(t23.to_string())

                              heavy index  kestrel index  heavy panelists watching  kestrel panelists watching
label                                                                                                         
Dancing with the Stars (ABC)         96.8           82.7                       545                         117
Grotesquerie (FX)                    95.9          129.6                       185                          41
Halloween Wars (FOOD)               116.2           95.2                        94                          22
Hannity (FNC)                        77.1           42.6                       114                          18
Monday Night Football (ABC)          93.6           85.9                       546                         127
NCIS (CBS)                          110.1           56.5                       211                          31
Sunday Night Football (NBC)          99.3           98.5                       733                         157
U

**Answer:** The Kestrel index jumps (42.6 for Hannity to 129.6 for Grotesquerie) because it rests on far fewer people: only 354 Kestrel buyers in the panel versus 1,661 heavy buyers, and per program only 18 (Hannity) to 41 (Grotesquerie) Kestrel panelists watched versus 94 to 733 heavy buyers. When a rating is built on a few dozen people, each representing tens of thousands, one or two viewers more or fewer move the index a lot. The heavy-buyer index (77.1 to 126.8) is more stable. I would trust the heavy-buyer index. To check: (1) bootstrap the panelists and report a confidence interval for each index; (2) compare the index by week or by random halves of the panel and see whether it is stable; (3) since the fusion links buying to viewing only through age, sex and income, predict each program's expected index from its demographic mix and treat large departures from it as noise.

## Task 3.1 — Weights

In [9]:
for n, d in [("acr_sessions", acr_sessions), ("hh_attributes", hh_attributes), ("reference_ratings", reference_ratings)]:
    print(n, d.shape); print(d.dtypes); print(d.head(8).to_string()); print()

acr_sessions (6433473, 9)
session_id                    int64
tv_id                         int64
hashed_ip                    object
session_type                 object
network_code                 object
input_source                 object
start_utc       datetime64[us, UTC]
end_utc         datetime64[us, UTC]
dma_code                      int64
dtype: object
   session_id    tv_id         hashed_ip session_type network_code   input_source                 start_utc                   end_utc  dma_code
0   900000001  7000360  59ef26c76b07e8d2         live          MTV      cable_box 2024-10-14 10:00:00+00:00 2024-10-14 10:06:27+00:00       641
1   900000002  7000382  189ea07d1cd85aea         live         ESPN      cable_box 2024-10-14 10:00:00+00:00 2024-10-14 10:04:21+00:00       602
2   900000003  7000449  488e94fe8965aa77         live         TOON  satellite_box 2024-10-14 10:00:00+00:00 2024-10-14 10:19:26+00:00       532
3   900000004  7002143  a1bc8b9e73b7fd48         live       

In [10]:
import numpy as np
ET = "America/New_York"
acr = acr_sessions.copy()
acr["start"] = acr.start_utc.dt.tz_convert(ET).dt.tz_localize(None)
acr["end"] = acr.end_utc.dt.tz_convert(ET).dt.tz_localize(None)
print(acr.start.min(), acr.start.max())
print(acr.session_type.value_counts()); print(acr.input_source.value_counts())
print(hh_attributes.head_age_band.value_counts())
print(hh_attributes.groupby("hashed_ip").household_id.nunique().value_counts())
print(acr.groupby("tv_id").hashed_ip.nunique().value_counts())

2024-10-14 06:00:00 2024-10-28 05:59:56
session_type
live       3463018
unknown    1776522
app        1130185
dvr          63748
Name: count, dtype: int64
input_source
cable_box           2462564
built_in_app        1574166
streaming_device     913682
antenna              748557
satellite_box        681270
game_or_disc          53234
Name: count, dtype: int64
head_age_band
35-49    15818
65+      14591
50-64    13631
18-34    10236
Name: count, dtype: int64
household_id
1    49153
2      527
3      511
4      353
5      184
6       34
Name: count, dtype: int64
hashed_ip
1    29981
Name: count, dtype: int64


In [11]:
LIVE, MIN_SEC = "live", 60
acr = acr[(acr.end - acr.start).dt.total_seconds() >= MIN_SEC].copy()
acr["bday"] = (acr.start - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")

tvip = acr[["tv_id", "hashed_ip"]].drop_duplicates()
tvip = tvip[tvip.groupby("tv_id").hashed_ip.transform("nunique") == 1]
hn = hh_attributes.groupby("hashed_ip").household_id.nunique()
hh1 = hh_attributes[hh_attributes.hashed_ip.isin(hn[hn == 1].index)]
tv = tvip.merge(hh1[["hashed_ip", "household_id", "income_band", "head_age_band"]], on="hashed_ip")
tv["head_age"] = np.where(tv.head_age_band == "65+", "65+", "Under 65")
print("TVs matched to one household:", tv.tv_id.nunique())

intab = acr[["tv_id", "bday"]].drop_duplicates().merge(tv[["tv_id", "head_age", "income_band"]], on="tv_id")
cnt = (intab.groupby(["bday", "head_age", "income_band"]).tv_id.nunique().rename("n").reset_index()
       .merge(universe, on=["head_age", "income_band"]))
cnt["weight"] = cnt.tv_households / cnt.n
intab = intab.merge(cnt[["bday", "head_age", "income_band", "weight"]], on=["bday", "head_age", "income_band"])
wsum = intab.groupby("bday").weight.sum() / 1e6
print("Weights sum per day (M):", round(wsum.min(), 1), "-", round(wsum.max(), 1),
      "| universe (M):", round(universe.tv_households.sum() / 1e6, 1))

ref = reference_ratings.merge(programs[["telecast_id", "network_code", "program_name", "broadcast_date", "start_ts", "end_ts"]],
                              on="telecast_id")
rows = []
for _, r in ref.iterrows():
    s = acr[(acr.network_code == r.network_code) & (acr.session_type == LIVE) &
            (acr.start < r.end_ts) & (acr.end > r.start_ts)]
    ov = (s.end.clip(upper=r.end_ts) - s.start.clip(lower=r.start_ts)).dt.total_seconds()
    watched = set(ov.groupby(s.tv_id).sum().loc[lambda v: v >= MIN_SEC].index)
    base = intab[intab.bday == str(r.broadcast_date)]
    w = base[base.tv_id.isin(watched)]
    rows.append((r.telecast_id, f"{r.program_name} ({r.network_code})",
                 round(len(w) / len(base) * 100, 2), round(w.weight.sum() / base.weight.sum() * 100, 2), r.hh_rating))
t3 = pd.DataFrame(rows, columns=["telecast_id", "telecast", "unweighted", "weighted", "reference"])
gu = (t3.unweighted - t3.reference).abs().mean(); gw = (t3.weighted - t3.reference).abs().mean()
print(t3.to_string(index=False))
print("Mean gap unweighted (points):", round(gu, 2))
print("Mean gap weighted (points):", round(gw, 2))
print("Share of gap closed:", round(1 - gw / gu, 3))

TVs matched to one household: 19607
Weights sum per day (M): 125.8 - 125.8 | universe (M): 125.8
 telecast_id                                     telecast  unweighted  weighted  reference
     1013636                         MLB Postseason (FOX)       17.03     17.76      25.86
     1021823                           NFL Football (NBC)       22.62     23.75      33.14
     1004042                                   NCIS (CBS)        5.50      5.80      10.56
     1012442                                Hannity (FNC)        1.04      1.35       2.30
     1016852                     Unsellable Houses (HGTV)        2.12      2.08       3.69
     1013031         Halloween Baking Championship (FOOD)        0.95      0.90       1.63
     1014236                            Grotesquerie (FX)        2.17      2.11       3.53
     1003476 The Real Housewives of New York City (BRAVO)        0.71      0.64       1.23
Mean gap unweighted (points): 3.72
Mean gap weighted (points): 3.44
Share of gap clo

In [12]:
u = universe.groupby("head_age").tv_households.sum()
cmp = pd.DataFrame({"smart-TV share": tv.drop_duplicates("tv_id").groupby("head_age").size() / tv.tv_id.nunique(),
                    "universe share": u / u.sum()})
print(cmp.round(3))

          smart-TV share  universe share
head_age                                
65+                0.194           0.283
Under 65           0.806           0.717


## Task 3.2 — What weighting cannot fix

**Answer:** Weighting moved the four broad-appeal programs up (NFL Football NBC +1.13 points, MLB Postseason FOX +0.73, Hannity +0.31, NCIS +0.30) and the four niche cable programs down slightly (Real Housewives -0.07, Grotesquerie -0.06, Halloween Baking Championship -0.05, Unsellable Houses -0.04). The table above shows why: households with a head aged 65+ have a smaller share among the matched smart TVs than in the universe, so weighting gives each of them more weight. The programs that gained are the ones where, in this data, those households appear to matter most (broadcast sports, a network drama and news), while the cable programs lost a little because the households that were scaled down watch them slightly more. Still, the mean gap to the reference fell only from 3.72 to 3.44 points, so weighting closed just 7.6% of it. Most of the gap remains: every weighted rating is still far below the reference, which is a problem of level, not of mix. Weighting cannot fix what the smart-TV data does not observe: TVs without ACR and households that could not be matched to exactly one household, viewing recorded as "unknown", "app" or "dvr" instead of live, and the fact that households with smart TVs that report their viewing may watch differently from other households in the same cell (Lab 3, Lecture 4: weighting corrects only for the variables used and assumes that sampled and unsampled homes in a cell behave alike). Income barely moves these ratings because the income cells are already close to the universe, so weights shift little between them, and within an age group households of different incomes watch these programs at similar rates.

## Task 4.1 — October predicts October

In [13]:
dwa = daily_weights[daily_weights.resp_type.isin(["R", "L"])].copy()
dwa["RESPID"] = dwa.respondent_id.astype(str)
dwa["age"] = dwa.RESPID.map(age_s); dwa = dwa[dwa.age >= 2].copy()
dwa["broadcast_date"] = dwa.broadcast_date.astype(str)
dwa["heavy"] = dwa.RESPID.map(people.heavy).fillna(False).astype(bool)
size4 = {"p2": dwa.groupby("broadcast_date").weight.sum(),
         "heavy": dwa[dwa.heavy].groupby("broadcast_date").weight.sum()}

NETS = ["CBS", "NBC", "ABC", "FOX", "FNC"]
t4 = programs[(programs.daypart == "Prime") & programs.network_code.isin(NETS)].copy()
t4["broadcast_date"] = t4.broadcast_date.astype(str)
t4["dur"] = (t4.end_ts - t4.start_ts).dt.total_seconds() / 60

vv = viewings[viewings.resp_type.isin(["R", "L"]) & viewings.network_code.isin(NETS)][["respondent_id", "network_code", "start_ts", "end_ts"]].copy()
vv["broadcast_date"] = vv.start_ts.dt.strftime("%Y-%m-%d")
xx = (t4[["telecast_id", "network_code", "broadcast_date", "start_ts", "end_ts"]]
      .rename(columns={"start_ts": "t0", "end_ts": "t1"})
      .merge(vv, on=["network_code", "broadcast_date"]))
xx = xx[(xx.start_ts < xx.t1) & (xx.end_ts > xx.t0)]
xx["mins"] = ((xx[["end_ts", "t1"]].min(axis=1) - xx[["start_ts", "t0"]].max(axis=1)).dt.total_seconds() / 60)
xx = xx.merge(dwa[["respondent_id", "broadcast_date", "weight", "heavy"]], on=["respondent_id", "broadcast_date"])
xx["wm"] = xx.weight * xx.mins
x4 = xx

r4 = t4[["telecast_id", "network_code", "program_name", "broadcast_date", "dur"]].copy()
r4["wm_p2"] = r4.telecast_id.map(x4.groupby("telecast_id").wm.sum()).fillna(0)
r4["wm_heavy"] = r4.telecast_id.map(x4[x4.heavy].groupby("telecast_id").wm.sum()).fillna(0)
for k, c in [("p2", "wm_p2"), ("heavy", "wm_heavy")]:
    r4[f"rat_{k}"] = r4[c] / r4.dur / r4.broadcast_date.map(size4[k]) * 100

hist = r4[r4.broadcast_date <= "2024-10-21"]
fc = r4[(r4.broadcast_date >= "2024-10-22") & (r4.broadcast_date <= "2024-10-28")].copy()
key = ["network_code", "program_name"]
hm = hist.groupby(key)[["rat_p2", "rat_heavy"]].mean().rename(columns={"rat_p2": "f_p2", "rat_heavy": "f_heavy"}).reset_index()
fc = fc.merge(hm, on=key, how="inner")

def mape(a, f):
    m = a > 0
    return float((abs(a[m] - f[m]) / a[m]).mean() * 100)
fc["err_p2"] = (fc.rat_p2 - fc.f_p2).abs() / fc.rat_p2
fc["err_heavy"] = (fc.rat_heavy - fc.f_heavy).abs() / fc.rat_heavy
print("Telecasts forecast (n):", len(fc))
print("MAPE, P2+ (%):", round(mape(fc.rat_p2, fc.f_p2), 1))
print("MAPE, heavy buyers (%):", round(mape(fc.rat_heavy, fc.f_heavy), 1))
b = fc[fc.rat_p2 > 0].sort_values("err_p2", ascending=False).iloc[0]
print("Biggest miss, P2+ (program, date):", f"{b.program_name} ({b.network_code})", b.broadcast_date)

/tmp/ipykernel_4866/909131717.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dwa["heavy"] = dwa.RESPID.map(people.heavy).fillna(False).astype(bool)


Telecasts forecast (n): 101
MAPE, P2+ (%): 36.4
MAPE, heavy buyers (%): 39.0
Biggest miss, P2+ (program, date): Krapopolis (FOX) 2024-10-27


## Task 4.2 — Why it missed

In [14]:
top = fc[fc.rat_p2 > 0].sort_values("err_p2", ascending=False).head(5)
print(top[["program_name", "network_code", "broadcast_date", "f_p2", "rat_p2", "err_p2"]].round(3).to_string(index=False))
for _, r in top.iterrows():
    d = t4[(t4.network_code == r.network_code) & (t4.broadcast_date == r.broadcast_date)]
    print(r.network_code, r.broadcast_date, "->", d.sort_values("start_ts").program_name.tolist())
print("heavy panelists per telecast, median:", x4[x4.heavy].groupby("telecast_id").respondent_id.nunique().median())

        program_name network_code broadcast_date  f_p2  rat_p2  err_p2
          Krapopolis          FOX     2024-10-27 1.062   0.218   3.878
Universal Basic Guys          FOX     2024-10-27 0.676   0.258   1.622
        Fire Country          CBS     2024-10-27 1.698   0.666   1.548
             Accused          FOX     2024-10-22 0.950   0.373   1.546
  What Would You Do?          ABC     2024-10-23 0.438   0.184   1.381
FOX 2024-10-27 -> ['Local / Syndicated Programming', 'The Simpsons', 'Universal Basic Guys', "Bob's Burgers", 'Krapopolis', "Bob's Burgers", 'Krapopolis']
FOX 2024-10-27 -> ['Local / Syndicated Programming', 'The Simpsons', 'Universal Basic Guys', "Bob's Burgers", 'Krapopolis', "Bob's Burgers", 'Krapopolis']
CBS 2024-10-27 -> ['60 Minutes', "Poppa's House", 'Fire Country', 'Tracker', 'The Equalizer', 'Fire Country']
FOX 2024-10-22 -> ['Murder in a Small Town', 'Accused', 'Universal Basic Guys', 'Murder in a Small Town']
ABC 2024-10-23 -> ['The Golden Bachelorette', 'A

In [15]:
for _, r in top.iterrows():
    d = programs[(programs.network_code == r.network_code) & (programs.broadcast_date.astype(str) == r.broadcast_date)]
    print("\n", r.program_name, r.network_code, r.broadcast_date)
    print(d.sort_values("start_ts")[["start_ts", "end_ts", "daypart", "program_name", "genre"]].tail(14).to_string(index=False))
    h = programs[(programs.network_code == r.network_code) & (programs.program_name == r.program_name) &
                 (programs.daypart == "Prime") & (programs.broadcast_date.astype(str) <= "2024-10-21")]
    print("usual slots:", h.start_ts.dt.strftime("%a %H:%M").value_counts().head(3).to_dict())
print("P2+ panelists per telecast, median:", x4.groupby("telecast_id").respondent_id.nunique().median())


 Krapopolis FOX 2024-10-27
           start_ts              end_ts      daypart                   program_name     genre
2024-10-27 11:00:00 2024-10-27 12:00:00      Daytime                FOX NFL Kickoff    Sports
2024-10-27 12:00:00 2024-10-27 13:00:00      Daytime                 FOX NFL Sunday Talk Show
2024-10-27 13:00:00 2024-10-27 16:05:00      Daytime                   NFL Football    Sports
2024-10-27 16:05:00 2024-10-27 19:15:00 Early Fringe                   NFL Football    Sports
2024-10-27 19:15:00 2024-10-27 20:00:00        Prime Local / Syndicated Programming   General
2024-10-27 20:00:00 2024-10-27 20:30:00        Prime                   The Simpsons    Comedy
2024-10-27 20:30:00 2024-10-27 21:00:00        Prime           Universal Basic Guys    Comedy
2024-10-27 21:00:00 2024-10-27 21:30:00        Prime                  Bob's Burgers    Comedy
2024-10-27 21:30:00 2024-10-27 22:00:00        Prime                     Krapopolis    Comedy
2024-10-27 22:00:00 2024-10-27 2

**Answer:** The biggest misses are on Sunday 2024-10-27 and on programs with only a few earlier airings. On 2024-10-27 the NFL pushed the schedule: on CBS, Fire Country was cut to a 14-minute and a 16-minute fragment (20:30-20:44 and 22:44-23:00) and the whole evening shifted, and on FOX the evening started after a local-programming gap and Krapopolis aired twice in prime (21:30 and 22:30), while the forecast is the mean of full-length airings in other slots (usual slots: Wednesday 22:00 and Sunday 21:30). Accused (FOX, 10-22) and What Would You Do? (ABC, 10-23) were forecast from only 1-3 earlier airings, in different days or slots, with different lead-ins, and the actual ratings were low (0.37 and 0.18), so a small absolute miss gives a large percentage error. The heavy-buyer forecast is worse (MAPE 39.0% vs 36.4%) because it rests on far fewer people: the median telecast has 46 heavy-buyer panelists against 244 persons 2+ panelists, so sampling noise is bigger (Lecture 6, error vs. sample size). Two improvements: (1) forecast from the same program in the same day and time slot with its lead-in, and adjust for known schedule changes such as sports overruns and shortened or doubled airings; (2) forecast the heavy-buyer rating as the P2+ forecast times a stable heavy-buyer share built from many telecasts (borrowing strength from the larger sample), instead of from the few heavy buyers watching one program.

## Task 5.1 — The plan

In [16]:
import itertools
import numpy as np

BUDGET = 1_000_000
p = out[["cost per :30 ($)", "target CPM ($)", "target viewers (k)", "composition"]].copy()
p["cap"] = (res.groupby("label").telecast_id.count() / 4.4).round().clip(lower=1).astype(int)
names = list(p.index)
cost = p["cost per :30 ($)"].to_numpy(dtype=float)
imp = (p["target viewers (k)"] * 1000).to_numpy(dtype=float)

best_imp, best_combo = 0.0, np.zeros(len(names), dtype=int)
for combo in itertools.product(*[range(int(c) + 1) for c in p["cap"]]):
    c = np.array(combo)
    if (c * cost).sum() <= BUDGET and (c * imp).sum() > best_imp:
        best_imp, best_combo = (c * imp).sum(), c

plan = pd.DataFrame({"spots": best_combo,
                     "cost ($)": (best_combo * cost).astype(int),
                     "target impressions": (best_combo * imp).round(0),
                     "composition": p["composition"].to_numpy()}, index=names)
plan = plan[plan.spots > 0]
print(plan.to_string())
print("Plan cost ($):", int(plan["cost ($)"].sum()))
print("Plan target impressions (M):", round(plan["target impressions"].sum() / 1e6, 1))
adult_imp = (plan["target impressions"] / plan.composition).sum()
print("Share of impressions that are heavy buyers:", round(plan["target impressions"].sum() / adult_imp, 3))

                              spots  cost ($)  target impressions  composition
Dancing with the Stars (ABC)      2    317200           3128400.0        0.191
Grotesquerie (FX)                 6     45552            745200.0        0.188
Halloween Wars (FOOD)             2     19400            323800.0        0.228
Hannity (FNC)                     2     79800            833800.0        0.152
Sunday Night Football (NBC)       1    487200           2865000.0        0.195
Unsellable Houses (HGTV)          3     38700            951600.0        0.249
Plan cost ($): 987852
Plan target impressions (M): 8.8
Share of impressions that are heavy buyers: 0.193


## Task 5.2 — Recommendation

**Answer:** I recommend the plan in Task 5.1: 2 spots on Dancing with the Stars (ABC), 1 on Sunday Night Football (NBC), 2 on Hannity (FNC), 6 on Grotesquerie (FX), 2 on Halloween Wars (FOOD) and 3 on Unsellable Houses (HGTV), at \$987,852 for 8.8M heavy-buyer impressions, an average target CPM of about \$112. The three cheap cable programs (target CPM \$41-\$61) are the efficient part, but together they deliver only about 2.0M impressions, so they cannot absorb \$1M alone; the volume comes from Dancing with the Stars and Sunday Night Football (68% of the impressions) at target CPMs of \$101 and \$170. I did not buy Monday Night Football or NCIS because their target CPMs (\$174 and \$148) are among the highest, and I did not chase the highest index (Unsellable Houses, 126.8), because it cannot deliver volume. The cap on spots per program is my assumption: its October telecasts divided by 4.4 weeks. Delivery is uncertain: the per-telecast forecast error was 39.0% MAPE for heavy buyers (36.4% for persons 2+), and the largest errors came from schedule changes such as the NFL Sunday, so a single spot can easily miss, although errors across 16 spots partly cancel; the cable programs were not covered by the Task 4 forecast at all. Composition is 0.193, so only about 19% of impressions are heavy buyers and roughly \$797k of the \$988k reaches other people; this is the price of buying programs instead of the audience itself. After the week I would compare the delivered heavy-buyer impressions per spot with this plan (and ask for make-goods where a guaranteed spot falls short), check the schedule for overruns and pre-emptions, and measure whether Kestrel purchases rose among exposed households.